In [22]:
import random
from collections import Counter
from pathlib import Path
import re
from openai import OpenAI
import os
from tqdm import tqdm


### set up data
https://archive.ics.uci.edu/dataset/228/sms+spam+collection

In [8]:
rows = []
with Path("/Users/zhengzikun/Desktop/MLDS 424/project/data/SMSSpamCollection").open("r", encoding="utf-8") as f:
    for line in f:
        label, text = line.rstrip("\n").split("\t", 1)
        rows.append({"label": label, "text": text})


In [9]:
rows[:3]

[{'label': 'ham',
  'text': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'},
 {'label': 'ham', 'text': 'Ok lar... Joking wif u oni...'},
 {'label': 'spam',
  'text': "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"}]

# 1. Find task/data and create scoring method

In [11]:
SEED = 42
N_EVAL = 350

random.seed(SEED)

# split by class
ham = [r for r in rows if r["label"] == "ham"]
spam = [r for r in rows if r["label"] == "spam"]

# class proportions from full data
p_spam = len(spam) / len(rows)

n_spam = round(N_EVAL * p_spam)
n_ham = N_EVAL - n_spam

eval_spam = random.sample(spam, n_spam)
eval_ham  = random.sample(ham, n_ham)

eval_set = eval_spam + eval_ham
random.shuffle(eval_set)

print("Full counts:", Counter([r["label"] for r in rows]))
print("Eval counts:", Counter([r["label"] for r in eval_set]))
print("Eval size:", len(eval_set))

Full counts: Counter({'ham': 4827, 'spam': 747})
Eval counts: Counter({'ham': 303, 'spam': 47})
Eval size: 350


In [12]:
# remaining data after removing eval_set
eval_ids = set(id(r) for r in eval_set)
remaining = [r for r in rows if id(r) not in eval_ids]

N_OPT = 250

random.seed(SEED + 1)

# stratified again
ham_rem = [r for r in remaining if r["label"] == "ham"]
spam_rem = [r for r in remaining if r["label"] == "spam"]

p_spam_rem = len(spam_rem) / len(remaining)

n_spam_opt = round(N_OPT * p_spam_rem)
n_ham_opt = N_OPT - n_spam_opt

opt_spam = random.sample(spam_rem, n_spam_opt)
opt_ham  = random.sample(ham_rem, n_ham_opt)

opt_set = opt_spam + opt_ham
random.shuffle(opt_set)

print("Opt counts:", Counter([r["label"] for r in opt_set]))
print("Opt size:", len(opt_set))


Opt counts: Counter({'ham': 217, 'spam': 33})
Opt size: 250


I used the UCI **SMS Spam Collection** dataset to define a binary classification task: given an SMS message, predict whether it is **spam** or **ham**. Ground-truth labels are provided by the dataset. I created a **stratified evaluation set** of 350 messages (ham=303, spam=47) to preserve the original class proportion, and a separate **stratified optimization/training set** of 250 messages (ham=217, spam=33) reserved for automated prompt engineering to avoid tuning on the evaluation set. The evaluation metric will be **accuracy**, and model outputs will be parsed from a required final-line format `#### spam` or `#### ham`.

### Base Prompt (No Prompt Engineering)

Template:

You are given an SMS message. Classify it as spam or ham.

Output the final answer on the last line with prefix #### followed by only one word: spam or ham.

SMS: {sms_text}


In [13]:
BASE_PROMPT = """You are given an SMS message. Classify it as spam or ham.

Output the final answer on the last line with prefix #### followed by only one word: spam or ham.

SMS: {sms_text}
"""


In [16]:
def parse_label(llm_text: str):
    """
    Extract 'spam' or 'ham' from the model output.
    Required format: last line contains '#### spam' or '#### ham' (case-insensitive).
    Returns: 'spam'/'ham' if found, else None.
    """
    if llm_text is None:
        return None

    # look for the last occurrence of #### ...
    matches = re.findall(r"####\s*(\w+)", llm_text, flags=re.IGNORECASE)
    if not matches:
        return None

    label = matches[-1].strip().lower()
    if label in {"spam", "ham"}:
        return label
    return None


In [17]:
print(parse_label("blah blah\n#### spam"))
print(parse_label("Reasoning...\n#### HAM"))
print(parse_label("#### unsure"))
print(parse_label("no delimiter here"))


spam
ham
None
None


I implemented a robust parser to extract the predicted label from LLM outputs using a required delimiter format: the final answer must appear as `#### spam` or `#### ham` (case-insensitive). If the delimiter is missing or the label is invalid, the prediction is treated as incorrect (returns `None`), enabling reliable automated scoring.

In [20]:
#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
os.environ["OPENAI_API_KEY"] = "##############"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"

def call_llm(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
    )
    return resp.choices[0].message.content


In [21]:
sample = eval_set[0]
prompt = BASE_PROMPT.format(sms_text=sample["text"])
out = call_llm(prompt)
print("PARSED:", parse_label(out))
print("GT:", sample["label"])


PARSED: ham
GT: ham


I implemented an automated scoring pipeline that (1) formats each SMS into a prompt, (2) queries an LLM (GPT-4o-mini), (3) parses the model’s predicted label from a required delimiter format (`#### spam` / `#### ham`), and (4) compares the parsed label against the dataset ground-truth label to compute correctness. A smoke test confirmed the parsed prediction matched the ground truth on a sample example.

In [23]:
def score_prompt(prompt_template: str, dataset):
    correct = 0
    total = len(dataset)
    bad_parse = 0

    for row in tqdm(dataset):
        prompt = prompt_template.format(sms_text=row["text"])
        out = call_llm(prompt)
        pred = parse_label(out)

        if pred is None:
            bad_parse += 1
            is_correct = False
        else:
            is_correct = (pred == row["label"])

        correct += int(is_correct)

    return {
        "accuracy": correct / total,
        "correct": correct,
        "total": total,
        "bad_parse": bad_parse
    }


In [24]:
baseline_results = score_prompt(BASE_PROMPT, eval_set)
baseline_results

100%|██████████| 350/350 [03:10<00:00,  1.84it/s]


{'accuracy': 0.9314285714285714, 'correct': 326, 'total': 350, 'bad_parse': 0}

Using the base prompt (no prompt engineering) and an exact-match parser on the required final-line format `#### {spam|ham}`, I evaluated GPT-4o-mini on a stratified evaluation set of 350 SMS messages. The baseline accuracy was about **93.14%** (326/350), with **0 parsing failures** (`bad_parse = 0`).

# 2. Create manual prompt engineering improvements and examine change in score

### Best-practice manual prompt improvement (role + specific instructions)

I improve the base prompt by:
- specifying the model role for more consistent outputs
- adding more detailed, specific classification guidance



In [25]:
IMPROVED_PROMPT_B1 = """You are an expert SMS spam detector.

Task: Classify the SMS message as spam or ham.

Use these criteria:
- spam: unsolicited promotions, prizes/lotteries, urgent calls to action, requests for money, suspicious links/numbers, or attempts to trick the user.
- ham: normal personal, conversational, or legitimate informational messages.

Output the final answer on the last line with prefix #### followed by only one word: spam or ham.

SMS: {sms_text}
"""


In [26]:
improved_b1_results = score_prompt(IMPROVED_PROMPT_B1, eval_set)
improved_b1_results


100%|██████████| 350/350 [06:41<00:00,  1.15s/it]


{'accuracy': 0.9428571428571428, 'correct': 330, 'total': 350, 'bad_parse': 0}

When evaluated on the 350-message stratified evaluation set, the improved prompt achieved an accuracy of about **94.29%** (330/350), compared to the baseline accuracy of about **93.14%** (326/350). This demonstrates that clearer role specification and more explicit task instructions improved model performance.

## Few-shot prompting

I add a small number of labeled examples (from the separate optimization set, not the evaluation set) to demonstrate the desired input-output pattern for spam vs ham classification.

In [27]:
SEED_FEWSHOT = 42

def build_fewshot_prompt(opt_set, k_ham=3, k_spam=3, seed=SEED_FEWSHOT):
    random.seed(seed)
    ham = [r for r in opt_set if r["label"] == "ham"]
    spam = [r for r in opt_set if r["label"] == "spam"]

    ex_ham = random.sample(ham, k_ham)
    ex_spam = random.sample(spam, k_spam)

    examples = ex_ham + ex_spam
    random.shuffle(examples)

    # Format examples to match the required parsing rule
    ex_blocks = []
    for ex in examples:
        ex_blocks.append(
            f"SMS: {ex['text']}\n#### {ex['label']}"
        )

    examples_text = "\n\n".join(ex_blocks)

    FEWSHOT_PROMPT = f"""You are an expert SMS spam detector.

Below are labeled examples. Use them to classify the new SMS as spam or ham.

{examples_text}

Now classify the following SMS.

Output the final answer on the last line with prefix #### followed by only one word: spam or ham.

SMS: {{sms_text}}
"""
    return FEWSHOT_PROMPT


In [28]:
FEWSHOT_PROMPT_B2 = build_fewshot_prompt(opt_set)


In [29]:
fewshot_results = score_prompt(FEWSHOT_PROMPT_B2, eval_set)
fewshot_results


100%|██████████| 350/350 [02:47<00:00,  2.09it/s]


{'accuracy': 0.9628571428571429, 'correct': 337, 'total': 350, 'bad_parse': 0}

### Conclusion for manual prompt engineering improvements

I implemented few-shot prompting by adding 6 labeled examples (3 ham, 3 spam) sampled from a separate optimization set (not the evaluation set). These examples explicitly demonstrate the correct input-output format and classification behavior. Labeled examples are provided in-context to guide the model’s decision boundaries.

When evaluated on the same 350-message stratified evaluation set:

- Baseline accuracy: ~93.14% (326/350)
- Best-practice prompt: ~94.29% (330/350)
- Few-shot prompt: ~96.29% (337/350)

Few-shot prompting produced the largest improvement, increasing accuracy by ~3.15 percentage points over the baseline. This demonstrates that providing concrete labeled examples significantly improves classification performance.


# 3. Implement an automated prompt engineering method and examine change in score

In [30]:
random.seed(42)

SEED_OPRO = 42


In [31]:
TASK_INSTRUCTIONS = """Classify the SMS message as spam or ham."""

OUTPUT_FORMAT_RULE = """Output the final answer on the last line with prefix #### followed by only one word: spam or ham."""

PROMPT_TEMPLATE_OPRO = """You are an expert SMS spam detector.

Task: {task_instructions}

Optimized instructions:
{subprompt}

{output_format_rule}

SMS: {sms_text}
"""


In [32]:
def make_full_prompt(subprompt: str, sms_text: str) -> str:
    return PROMPT_TEMPLATE_OPRO.format(
        task_instructions=TASK_INSTRUCTIONS,
        subprompt=subprompt.strip(),
        output_format_rule=OUTPUT_FORMAT_RULE,
        sms_text=sms_text
    )

def make_optimizer_prompt(history, k_new=6):
    """
    history: list of dicts: [{"subprompt": "...", "acc": 0.94}, ...] sorted best->worst
    """
    hist_text = ""
    for i, h in enumerate(history[:10], 1):  # keep it short/cheap
        hist_text += f"\n[{i}] accuracy={h['acc']:.4f}\nSUBPROMPT:\n{h['subprompt']}\n"

    return f"""You are an optimizer for prompt instructions.

We are optimizing ONLY the 'Optimized instructions' block (called the optimized sub-prompt).
Task: SMS spam classification (labels: spam vs ham).

Constraints for the optimized sub-prompt:
- 3 to 7 bullet points
- Must be general (no examples, no few-shot)
- Focus on decision cues and edge cases
- Do NOT include the required output format line (we add that elsewhere)

Previous tried optimized sub-prompts and their accuracies on an optimization set:
{hist_text}

Generate {k_new} NEW optimized sub-prompts.
Return them in the following format exactly:

<CANDIDATE>
- bullet...
- bullet...
</CANDIDATE>
(repeat)
"""


In [33]:
def score_subprompt_on_opt(subprompt: str, opt_set):
    """
    - plug 'optimized sub-prompt' into a fixed prompt template
    - evaluate accuracy on the optimization/selection set (opt_set)
    """
    correct = 0
    bad_parse = 0
    total = len(opt_set)

    for row in tqdm(opt_set):
        full_prompt = make_full_prompt(subprompt=subprompt, sms_text=row["text"])
        out = call_llm(full_prompt)
        pred = parse_label(out)

        if pred is None:
            bad_parse += 1
            is_correct = False
        else:
            is_correct = (pred == row["label"])

        correct += int(is_correct)

    return {
        "acc": correct / total,
        "correct": correct,
        "total": total,
        "bad_parse": bad_parse,
        "subprompt": subprompt
    }


### mirrored the Lab 5 OPRO
checking if it will work

In [34]:
INIT_SUBPROMPT = """- Treat promotions, prizes, lotteries, and "free" offers as spam cues.
- Treat urgent calls to action and payment requests as spam cues.
- Treat suspicious links, short codes, or unknown numbers as spam cues.
- Otherwise, classify as ham."""
test_res = score_subprompt_on_opt(INIT_SUBPROMPT, opt_set)
test_res


100%|██████████| 250/250 [03:07<00:00,  1.34it/s]


{'acc': 0.988,
 'correct': 247,
 'total': 250,
 'bad_parse': 0,
 'subprompt': '- Treat promotions, prizes, lotteries, and "free" offers as spam cues.\n- Treat urgent calls to action and payment requests as spam cues.\n- Treat suspicious links, short codes, or unknown numbers as spam cues.\n- Otherwise, classify as ham.'}

I mirrored the Lab 5 OPRO outline by defining an *optimized sub-prompt* (an instruction block) inserted into a fixed prompt template. I implemented a scoring function that evaluates any candidate sub-prompt on a separate **optimization set** (opt_set, n=250) using accuracy with strict parsing (`#### spam|ham`). A smoke test sub-prompt achieved **98.8%** accuracy (247/250) with **0 parsing failures**, confirming the optimization-scoring pipeline works.

### Implement one OPRO iteration
generate candidates -> score -> update history

In [35]:
def parse_candidates(text: str):
    cands = re.findall(r"<CANDIDATE>\s*(.*?)\s*</CANDIDATE>", text, flags=re.DOTALL)
    cands = [c.strip() for c in cands if c.strip()]
    return cands

def opro_iteration(history, k_new=6):
    # build optimizer prompt from history
    opt_prompt = make_optimizer_prompt(history, k_new=k_new)
    raw = call_llm(opt_prompt)
    candidates = parse_candidates(raw)

    results = []
    for c in candidates:
        res = score_subprompt_on_opt(c, opt_set)
        results.append(res)

    # merge into history and keep best first
    new_history = history + results
    new_history = sorted(new_history, key=lambda d: d["acc"], reverse=True)

    return new_history, candidates, results, raw


In [36]:
history = [test_res]
history = sorted(history, key=lambda d: d["acc"], reverse=True)

history, candidates, results, raw_optimizer = opro_iteration(history, k_new=6)


100%|██████████| 250/250 [04:01<00:00,  1.03it/s]


In [37]:
print("Num candidates proposed:", len(candidates))
print("Top 3 accuracies this round:", [round(r["acc"], 4) for r in results[:3]])
print("Best overall acc so far:", history[0]["acc"])


Num candidates proposed: 6
Top 3 accuracies this round: [0.964, 0.952, 0.92]
Best overall acc so far: 0.988


### Final hold-out evaluation on eval_set (350)

In [39]:
best = history[0]
best_subprompt = best["subprompt"]

BEST_PROMPT_C = """You are an expert SMS spam detector.

Task: Classify the SMS message as spam or ham.

Optimized instructions:
{subprompt}

Output the final answer on the last line with prefix #### followed by only one word: spam or ham.

SMS: {sms_text}
"""


In [40]:
best_subprompt

'- Treat promotions, prizes, lotteries, and "free" offers as spam cues.\n- Treat urgent calls to action and payment requests as spam cues.\n- Treat suspicious links, short codes, or unknown numbers as spam cues.\n- Otherwise, classify as ham.'

In [41]:
BEST_PROMPT_C = BEST_PROMPT_C.format(subprompt=best_subprompt, sms_text="{sms_text}")

c_results = score_prompt(BEST_PROMPT_C, eval_set)


100%|██████████| 350/350 [04:24<00:00,  1.32it/s]


In [42]:
print("Selected best opt_set acc:", best["acc"])
print("Eval acc:", c_results["accuracy"], "| correct:", c_results["correct"], "/", c_results["total"], "| bad_parse:", c_results["bad_parse"])


Selected best opt_set acc: 0.988
Eval acc: 0.9571428571428572 | correct: 335 / 350 | bad_parse: 0


I implemented an OPRO-style automated prompt optimization procedure by optimizing an "instruction sub-prompt" within a fixed prompt template. Using a separate optimization set (opt_set, n=250), an optimizer LLM automatically generated 6 new candidate instruction blocks, each of which was scored by accuracy on opt_set. The best-performing sub-prompt on opt_set achieved about **98.8%** (247/250) with **0 parsing failures**, and this selected prompt was then evaluated once on a held-out evaluation set (eval_set, n=350) to avoid test-set leakage.

Final hold-out performance of the selected automated prompt was about **95.71%** (335/350), with **0 parsing failures**.

### Overall Accuracy Summary (Eval Set n=350)

- **Baseline prompt:** ~93.14% (326/350)
- **Best-practice:** ~94.29% (330/350)
- **Few-shot prompting:** ~96.29% (337/350)
- **Automated OPRO-style optimization:** ~95.71% (335/350)

All methods used a strict parseable output format (`#### spam` / `#### ham`) and achieved 0 parsing failures.

The results show that prompt engineering significantly affects LLM classification performance. Starting from a baseline accuracy of 93.14%, adding clearer role specification and detailed task instructions improved accuracy to 94.29%. Few-shot prompting produced the strongest performance at 96.29%, demonstrating that providing labeled in-context examples helps the model better understand decision boundaries between spam and ham messages. 

Interestingly, the automated OPRO-style optimization achieved 95.71%, which is slightly lower than the few-shot result. This outcome is reasonable because the automated method optimized only an instruction sub-prompt on the separate optimization set and did not include in-context labeled examples. Few-shot prompting directly provides concrete classification demonstrations, which can be particularly effective for pattern-recognition tasks like spam detection. In contrast, automated optimization focused on refining general decision rules, which may improve robustness but does not always outperform strong few-shot guidance. Overall, the experiments confirm that structured prompt engineering—both manual and automated—can meaningfully improve LLM performance, while also illustrating trade-offs between general instruction optimization and example-based conditioning.
